# Bài 5 – Trang 26: Phân lớp Heart Disease

**6451071011 - Lê Nguyễn Anh Dự**

Chạy các ô theo thứ tự trong Google Colab. Tệp `heart.csv` được tải từ liên kết Drive nếu chưa có trong thư mục làm việc. Vì đề không nêu tỷ lệ chia mới, dùng 70% huấn luyện và 30% kiểm thử, `random_state=42`, như bài trước.


In [ ]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

STUDENT = '6451071011 - Lê Nguyễn Anh Dự'
FILE_ID = '10N9jGzMn-MagDWXORdP-qIpwC3omZJHb'

def heading(number, title):
    print(STUDENT)
    print(f'Câu {number}. {title}')

def ensure_data():
    filename = 'heart.csv'
    if not os.path.exists(filename):
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
            import gdown
        gdown.download(id=FILE_ID, output=filename, quiet=False)
    if not os.path.exists(filename):
        raise FileNotFoundError('Không tải được heart.csv; hãy tải tệp lên Colab.')
    return filename


## Câu a. Nhập dữ liệu, xóa dòng trống và đổi tên cột

In [ ]:
heading('a', 'Nhập và tiền xử lý dữ liệu')
column_names = ['age', 'sex', 'chest_pain_type', 'resting_blood_pressure',
                'cholesterol', 'fasting_blood_sugar', 'rest_ecg',
                'max_heart_rate_achieved', 'exercise_induced_angina',
                'st_depression', 'st_slope', 'num_major_vessels',
                'thalassemia', 'target']
data = pd.read_csv(ensure_data(), encoding='utf-8-sig', skipinitialspace=True)
assert data.shape[1] == 14, f'Đã đọc {data.shape[1]} cột, dự kiến 14 cột.'
before = len(data)
data.columns = column_names
data = data.replace(r'^\s*\?\s*$', np.nan, regex=True)
data = data.replace(r'^\s*$', np.nan, regex=True)
for col in column_names:
    data[col] = pd.to_numeric(data[col], errors='coerce')
data = data.dropna().reset_index(drop=True)
data['target'] = data['target'].astype(int)
assert set(data['target'].unique()).issubset({0, 1}), 'target phải là 0 hoặc 1.'
print(f'Ban đầu: {before} dòng; sau làm sạch: {len(data)} dòng; đã xóa: {before-len(data)} dòng')
print('Tên cột:', data.columns.tolist())
display(data.head())


## Câu b.1. Khảo sát Pearson

Chỉ xét các thuộc tính đầu vào. Nếu hai cột có |r| ≥ 0,90, loại một cột trong cặp; không xét `target` để tránh bỏ cột theo nhãn.


In [ ]:
heading(1, 'Tương quan Pearson')
input_data = data.drop(columns='target')
corr = input_data.corr(method='pearson')
display(corr.round(3))
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Tương quan Pearson giữa các thuộc tính đầu vào')
plt.tight_layout(); plt.show()
upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] >= 0.90).any()]
print('Cột cần loại:', drop_corr if drop_corr else 'Không có')
input_data = input_data.drop(columns=drop_corr)


## Câu b.2. Tách thuộc tính và nhãn quyết định

In [ ]:
heading(2, 'Tách X và y')
X = input_data.copy()
y = data['target'].copy()
print('X:', X.shape, '| y:', y.shape)
print('Phân bố target (0: không bệnh; 1: có bệnh):')
display(y.value_counts().sort_index())


## Câu b.3. Mã hóa one-hot cột không phải số

In [ ]:
heading(3, 'One-hot encoding')
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
print('Cột không phải số:', categorical_columns if categorical_columns else 'Không có')
X = pd.get_dummies(X, columns=categorical_columns, dtype=np.uint8)
print('Kích thước sau mã hóa:', X.shape)
display(X.head())


## Câu b.4. Chia tập huấn luyện và kiểm thử

In [ ]:
heading(4, 'Chia dữ liệu 70/30')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('Huấn luyện:', len(X_train), 'mẫu')
print('Kiểm thử:', len(X_test), 'mẫu')


## Hàm đánh giá dùng chung

`DecisionTreeClassifier(criterion="entropy")` dùng information gain theo hướng dẫn. Với thuộc tính số, scikit-learn chia theo ngưỡng nên đây là mô phỏng ID3, không phải ID3 thuần túy.


In [ ]:
class_names = [0, 1]
results = []

def evaluate(name, model):
    pred = model.predict(X_test)
    print(classification_report(y_test, pred, labels=class_names,
                                target_names=['Không bệnh (0)', 'Có bệnh (1)'],
                                zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=class_names)
    print('Ma trận nhầm lẫn (hàng: thực tế; cột: dự đoán):')
    print(cm)
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['0', '1'], yticklabels=['0', '1'])
    plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.title(name)
    plt.tight_layout(); plt.show()
    results.append({'Mô hình': name,
                    'Accuracy': accuracy_score(y_test, pred),
                    'Precision (bệnh)': precision_score(y_test, pred, pos_label=1, zero_division=0),
                    'Recall (bệnh)': recall_score(y_test, pred, pos_label=1, zero_division=0),
                    'F1 (bệnh)': f1_score(y_test, pred, pos_label=1, zero_division=0)})

def show_tree(model, title, filename):
    print('Bốn tầng đầu của cây; mô hình được huấn luyện trên cây đầy đủ:')
    print(export_text(model, feature_names=list(X_train.columns), max_depth=3))
    plt.figure(figsize=(22, 10))
    plot_tree(model, feature_names=list(X_train.columns),
              class_names=['Không bệnh', 'Có bệnh'],
              filled=True, rounded=True, max_depth=3, fontsize=8)
    plt.title(title + ' – 4 tầng đầu')
    plt.tight_layout()
    plt.savefig(filename, dpi=100, bbox_inches='tight')
    plt.show()
    print('Đã lưu hình:', filename)


## Câu b.5. Cây ID3 theo entropy

In [ ]:
heading(5, 'Cây ID3/entropy')
id3 = DecisionTreeClassifier(criterion='entropy', random_state=42)
id3.fit(X_train, y_train)
evaluate('ID3 (entropy)', id3)
show_tree(id3, 'ID3 (entropy)', 'heart_tree_id3.png')


## Câu b.6. Cây CART theo Gini

In [ ]:
heading(6, 'Cây CART/Gini')
cart = DecisionTreeClassifier(criterion='gini', random_state=42)
cart.fit(X_train, y_train)
evaluate('CART (gini)', cart)
show_tree(cart, 'CART (gini)', 'heart_tree_cart.png')


## Câu b.7. Naïve Bayes

In [ ]:
heading(7, 'Gaussian Naïve Bayes')
nb = GaussianNB()
nb.fit(X_train, y_train)
evaluate('Gaussian Naïve Bayes', nb)


## Câu b.8. So sánh kết quả

In [ ]:
heading(8, 'So sánh ba mô hình')
comparison = pd.DataFrame(results).set_index('Mô hình')
display(comparison.style.format('{:.4f}').highlight_max(axis=0, color='#c8e6c9'))
for metric in ['Accuracy', 'Recall (bệnh)', 'F1 (bệnh)']:
    best = comparison[metric].idxmax()
    print(f'{metric} cao nhất: {best} ({comparison.loc[best, metric]:.4f})')
print('Nhận xét: Recall (bệnh) đo tỷ lệ người có bệnh được phát hiện; F1 (bệnh) cân bằng precision và recall. Accuracy cho biết tỷ lệ đúng chung. Cây không giới hạn độ sâu có nguy cơ quá khớp; GaussianNB giả định độc lập có điều kiện và phân phối Gaussian.')
